In [ ]:
import zipfile
import os
import yaml
from functools import lru_cache
import openai 

In [2]:
knwf_path = "test_data/2024_nuclei_segmentation_knime.knwf"
extract_dir = "test_data_unzipped"

1. Extract a KNIME .knwf file into the target directory.

In [3]:

def unzip_knime_workflow(zip_path: str, extract_to: str):
    """
    Entpackt eine KNIME .knwf Datei in das angegebene Zielverzeichnis.
    """
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Entpackt: {zip_path} → {extract_to}")

unzip_knime_workflow(knwf_path, extract_dir)

Entpackt: test_data/2024_nuclei_segmentation_knime.knwf → test_data_unzipped


2. Workflow-Verzeichnis finden

In [4]:
def find_workflow_dir(base_dir: str) -> str:
    """
    Sucht im Verzeichnis base_dir nach dem ersten Unterordner, der eine Datei namens 'workflow.knime' enthält.
    Gibt dann den vollständigen Pfad zu diesem Unterordner zurück.
    """
    for entry in os.listdir(base_dir):  # Alle Dateien und Ordner in base_dir durchgehen
        full_path = os.path.join(base_dir, entry)

        # Wenn es ein Ordner ist und darin 'workflow.knime' liegt → Erfolg
        if os.path.isdir(full_path) and "workflow.knime" in os.listdir(full_path):
            print(f"Workflow-Ordner gefunden: {full_path}")
            return full_path

    # Wenn kein solcher Ordner gefunden wurde → Fehler auslösen
    raise ValueError(f"⚠️ Kein gültiger Workflow-Ordner gefunden in {base_dir}")

workflow_dir = find_workflow_dir(extract_dir)


Workflow-Ordner gefunden: test_data_unzipped/2024_nuclei_segmentation_knime


3. Sammelt alle settings.xml-Dateien aus den Node-Unterordnern und gibt sie als dict zurück.
    Key: Node-Name (z. B. "Image Reader (#1)")
    Value: XML-String

In [5]:
def collect_knime_node_files(workflow_dir: str):
    """
    Sammelt alle settings.xml-Dateien aus den Node-Unterordnern und gibt sie als dict zurück.
    Key: Node-Name (z. B. "Image Reader (#1)")
    Value: XML-String
    """
    node_data = {}

    for entry in os.listdir(workflow_dir):
        full_path = os.path.join(workflow_dir, entry)
        if os.path.isdir(full_path) and not entry.startswith('.'):
            xml_path = os.path.join(full_path, "settings.xml")
            if os.path.exists(xml_path):
                with open(xml_path, 'r', encoding='utf-8') as f:
                    xml_content = f.read()
                    node_data[entry] = xml_content

    return node_data

In [6]:
knime_nodes = collect_knime_node_files(workflow_dir)
print("Erste 100 Zeichen jeder Node:")
for node_name, xml_content in knime_nodes.items():
    print(f"{node_name}: {xml_content[:100]}...")

Erste 100 Zeichen jeder Node:
Splitter (#2): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Gaussian Convolution (#3): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Reader (#1): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Features (#7): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Global Thresholder (#5): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Connected Component Analysis (#6): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
CLAHE (#4): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...


In [7]:
@lru_cache(maxsize=1)
def load_translation_examples(yaml_path: str = "data/translation_table.yml") -> list:
    with open(yaml_path, "r", encoding="utf-8") as f:
        docs = list(yaml.safe_load_all(f))
        print(docs)
        examples = []
        for doc in docs[0]:
            knime = doc.get("KNIME")
            galaxy = doc.get("Galaxy")

            if knime and galaxy:
                examples.append({
                    "KNIME": knime.strip(),
                    "Galaxy": galaxy.strip()
                })

    return examples


In [8]:
translation_examples = load_translation_examples()
translation_examples

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

[{'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">\n    <entry key="node_file" type="xstring" value="settings.xml"/>\n    <config key="flow_stack"/>\n    <config key="internal_node_subsettings">\n        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>\n    </config>\n    <config key="model">\n        <config key="check_file_format_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="EnabledStatus" type="xboolean" value="true"/>\n        </config>\n        <entry key="check_file_format" type="xboolean" value="true"/>\n        <config key="group_files_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="Enabled

In [9]:
def build_examples():
    examples_text =  """You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:
"""
    examples = load_translation_examples()
    if examples:
        for i, ex in enumerate(examples[:6]):  # z. B. nur 6 Beispiele
            examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return examples_text


In [10]:
examples = build_examples()
print(examples)

You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:


## Example 1:

KNIME node (XML):

```xml
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="check_file_format_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="check_file_format" type="xboolean" value="true"

In [13]:
def build_task_prompt(source: str) -> str:
    task_prompt = f"""
# Task
Convert the following KNIME node XML into a single **Galaxy workflow step**.

# Input
KNIME Node (XML):
```xml
{source}

# Output requirements
Respond with the Galaxy step ONLY as a valid JSON object conforming to Galaxy’s .ga workflow schema.

No extra text, no explanations, no comments, no surrounding markdown — just the JSON object.

Populate (at minimum): type, tool_id, tool_version (if known), label, tool_state (JSON-encoded string if required by Galaxy), inputs, outputs, position.

Map KNIME settings/ports to Galaxy parameters and connections. Preserve parameter names/values; convert datatypes/paths sensibly.

If information is missing, infer reasonable defaults to keep the step valid (do not include TODOs or comments).

Do not include the outer workflow wrapper — return the step object only as JSON.

# Assumptions and Mapping Rules
KNIME node parameters → Galaxy tool_state parameters (types converted appropriately).

KNIME input/output ports → Galaxy inputs / outputs entries with matching names.

Keep labels short and derived from the KNIME node name.

Use deterministic ordering of keys.
"""
    
    return task_prompt

In [14]:
task = build_task_prompt(knime_nodes["Image Reader (#1)"])
print(task)


# Task
Convert the following KNIME node XML into a single **Galaxy workflow step**.

# Input
KNIME Node (XML):
```xml
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="check_file_format_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="check_file_format" type="xboolean" value="true"/>
        <config key="group_files_Internals">
            <entry

In [15]:
full_prompt  = f"{examples}\n\n{task}"
print(full_prompt)

You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:


## Example 1:

KNIME node (XML):

```xml
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="check_file_format_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="check_file_format" type="xboolean" value="true"

In [16]:
def prompt_scadsai_llm(message:str, model="openai/gpt-oss-120b"):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    
    # convert message in the right format if necessary
    if isinstance(message, str):
        message = [{"role": "user", "content": message}]
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="https://llm.scads.ai/v1",
                           api_key=os.environ.get('SCADSAI_API_KEY')
    )
    response = client.chat.completions.create(
        model=model,
        messages=message
    )
    
    # extract answer
    return response.choices[0].message.content

In [17]:
answer = prompt_scadsai_llm(message= full_prompt)
print(answer)

{
    "type": "data_input",
    "tool_id": null,
    "tool_version": null,
    "label": "Image Reader",
    "tool_state": "{\"optional\": false, \"tag\": null}",
    "inputs": [
        {
            "description": "",
            "name": "Input Image Dataset"
        }
    ],
    "outputs": [],
    "position": {
        "left": 0.0,
        "top": 0.0
    },
    "id": 0,
    "annotation": "",
    "content_id": null,
    "errors": null,
    "name": "Input dataset",
    "uuid": "00000000-0000-0000-0000-000000000000",
    "when": null,
    "workflow_outputs": []
}
